# Generators & Iterators: Lazy Streaming

## What you'll practice

Generators let you process files bigger than memory, one record at a time, by chaining small stages — the same mental model
as Spark's lazy transformations. You'll write generator functions, a class-based iterator, and a streaming mini-pipeline.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pythonic/test_03_05_generators_iterators.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/03_pythonic_python/05_generators_iterators.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · Read meaningful lines lazily

**Implement:** `read_lines(path) -> Iterator[str]`

A generator that yields stripped lines from a text file, skipping blank lines and lines starting with `#` (after stripping). The file must be closed when iteration finishes.

**Input:** `path`

**Expected output:** `Iterator[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_read_lines` in `tests/pythonic/test_03_05_generators_iterators.py`.

**Check:** `pytest tests/pythonic/test_03_05_generators_iterators.py -k test_read_lines`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Batch any iterable

**Implement:** `batched(iterable, n: int) -> Iterator[tuple]`

Yield tuples of up to `n` items from **any** iterable (including infinite generators) — without `len()` or slicing. `n < 1` → `ValueError` (raised when called is fine, or on first `next`).

**Input:** `iterable, n: int`

**Expected output:** `Iterator[tuple]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
list(batched(range(5), 2))          # [(0, 1), (2, 3), (4,)]
from itertools import count
next(batched(count(), 3))            # (0, 1, 2)  -- works on infinite input
```

**Check:** `pytest tests/pythonic/test_03_05_generators_iterators.py -k "test_batched or test_batched_infinite or test_batched_invalid"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Tolerant JSON Lines parser

**Implement:** `iter_jsonl(lines, errors: list) -> Iterator[dict]`

Yield parsed objects from JSON Lines. Skip blank lines. For malformed lines, append `(line_no, error_message)` to the
caller-provided `errors` list and keep going. Non-object JSON (e.g. `[1,2]`, `"x"`) also counts as an error.
Try it on `datasets/json/events.jsonl` — about 1% of lines are truncated.

**Input:** `lines, errors: list`

**Expected output:** `Iterator[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_iter_jsonl, test_iter_jsonl_real_file` in `tests/pythonic/test_03_05_generators_iterators.py`.

**Check:** `pytest tests/pythonic/test_03_05_generators_iterators.py -k "test_iter_jsonl or test_iter_jsonl_real_file"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · A class-based paginating iterator

**Implement:** `class PageIterator`

`PageIterator(fetch_page, start_page=1)` iterates over **records** across pages. `fetch_page(page_number)` returns a list;
an empty list means no more pages. Implement `__iter__` and `__next__` (no generator functions inside!), fetch pages lazily
(only when the current page is exhausted), and expose `pages_fetched`. After exhaustion, `next()` keeps raising `StopIteration`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
pages = {1: ["a", "b"], 2: ["c"]}
it = PageIterator(lambda p: pages.get(p, []))
list(it), it.pages_fetched     # (['a', 'b', 'c'], 3)
```

**Check:** `pytest tests/pythonic/test_03_05_generators_iterators.py -k "test_page_iterator_lazy or test_page_iterator_start_page_and_empty"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Streaming event analytics on a large file · 🧩 Mini-project

**Implement:** `event_stats(path) -> dict`

Build a lazy pipeline over `datasets/json/events.jsonl` with **constant memory** (no list of all events):
`read lines → parse (iter_jsonl) → validate (user_id present, ts present) → aggregate`. Return:

```
{"lines": int, "malformed": int, "invalid": int, "valid": int,
 "by_type": {event_type: count, ...},            # sorted by count desc, then type
 "purchase_revenue": float,                       # sum of properties.amount for purchase events, 2 decimals
 "active_users": int,                             # distinct user_ids among valid events
 "first_ts": str | None, "last_ts": str | None}  # min/max ts among valid events
```
`lines` counts non-blank lines. Reuse your earlier generators where possible (define helpers here too if needed).
Test your function on a 200k-line file you generate yourself and watch memory stay flat.

**Input:** `path`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_event_stats_small, test_event_stats_empty, test_event_stats_real_and_large` in `tests/pythonic/test_03_05_generators_iterators.py`.

**Check:** `pytest tests/pythonic/test_03_05_generators_iterators.py -k "test_event_stats_small or test_event_stats_empty or test_event_stats_real_and_large"`

**YOUR CODE ↓**